# 28 · Phase H1d — Source and Feature Sensitivity Audit

This notebook investigates whether the H1c physical-state shift is broad physical-regime structure or disproportionately driven by a feature family / source-processing change.

It performs **no routing change and no policy selection**. All Cycle-25/2026 interpretation remains retrospective/post-hoc.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf

ROOT=Path("/home/abmoses2000")
H1B=ROOT/"graybox_aia72_physical_state_h1b_nonredundant_v1_20261006"
FULL=H1B/"physical_state_full_113433.csv.gz"
TRANSFORM=H1B/"physical_feature_transform.csv"
SHARP=ROOT/"graybox_phase_h_recovered_input_20261006"/"sharp.npy"
LEDGER=ROOT/"sharp72_predictions_frozen_20261002.csv.gz"
MANIFEST=ROOT/"graybox_phase_h_dataset_manifest_20261002.json"
SOURCE_ROOT=ROOT/"aia_sharp16_refit9_evaluation_preparation_20261002"/"inputs"/"sharp16_matched_v1_20261001"
OUT=ROOT/"graybox_aia72_physical_source_feature_h1d_v1_20261006"

for p in [FULL,TRANSFORM,SHARP,LEDGER,MANIFEST,SOURCE_ROOT/"build_receipt.json",SOURCE_ROOT/"independent_review.json"]:
    assert p.exists(), f"Missing required input: {p}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()

manifest=json.loads(MANIFEST.read_text())
features=manifest["feature_order"]
assert features==["MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD","TOTUSJZ","MEANALP","MEANJZH","ABSNJZH","SAVNCPP","MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT","TOTUSJH"]
assert sha256(SHARP)=="c9921be7b7fe381b43436a0e927d15d84de83e15376025ae0d8a79a6ef917639"
assert sha256(LEDGER)=="7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

X=np.load(SHARP,mmap_mode="r",allow_pickle=False)
ledger=pd.read_csv(LEDGER,low_memory=False).sort_values("tensor_row",kind="stable").reset_index(drop=True)
full=pd.read_csv(FULL,low_memory=False).sort_values("tensor_row",kind="stable").reset_index(drop=True)
transform=pd.read_csv(TRANSFORM)

assert X.shape==(113433,3,16)
assert len(ledger)==len(full)==len(X)
assert np.array_equal(ledger["tensor_row"].to_numpy(int),np.arange(len(X)))
assert np.array_equal(full["tensor_row"].to_numpy(int),np.arange(len(X)))

ledger["issue_dt"]=pd.to_datetime(ledger["issue_utc"],utc=True,errors="raise")
ledger["year"]=ledger["issue_dt"].dt.year.astype(int)
ledger["month"]=ledger["issue_dt"].dt.to_period("M").astype(str)
full["issue_dt"]=pd.to_datetime(full["issue_utc"],utc=True,errors="raise")
full["year"]=full["issue_dt"].dt.year.astype(int)
full["month"]=full["issue_dt"].dt.to_period("M").astype(str)

print("Verified H1d inputs.")
print("Tensor:",X.shape, X.dtype)
print("Source root:",SOURCE_ROOT)
print("Output:",OUT)


In [ ]:
# H1d-A: raw-feature monthly audit, latest slot only.
# We also express each monthly median in the exact H1b transformed/training-IQR coordinate.
tmap=transform.set_index("feature").to_dict(orient="index")
raw_rows=[]
for yr in [2024,2025,2026]:
    months=sorted(ledger.loc[ledger.year.eq(yr),"month"].unique())
    for month in months:
        mask=ledger["month"].eq(month).to_numpy()
        for j,f in enumerate(features):
            raw=np.asarray(X[mask,2,j],dtype=float)
            cfg=tmap[f]
            if cfg["transform"]=="log1p_abs":
                tt=np.log1p(np.abs(raw))
            elif cfg["transform"]=="log1p_nonnegative":
                tt=np.log1p(np.maximum(raw,0.0))
            elif cfg["transform"]=="identity":
                tt=raw.copy()
            else:
                raise ValueError(cfg["transform"])
            z=(tt-float(cfg["train_median"]))/float(cfg["train_iqr"])
            raw_rows.append({
              "month":month,"feature":f,"cases":int(mask.sum()),
              "raw_median":float(np.median(raw)),
              "raw_q25":float(np.quantile(raw,.25)),
              "raw_q75":float(np.quantile(raw,.75)),
              "transformed_median_training_iqr":float(np.median(z)),
              "transformed_q25_training_iqr":float(np.quantile(z,.25)),
              "transformed_q75_training_iqr":float(np.quantile(z,.75)),
            })
raw_monthly=pd.DataFrame(raw_rows)
raw_monthly.to_csv(OUT/"raw_feature_monthly_2024_2026.csv",index=False)

grad=raw_monthly[raw_monthly.feature.isin(["MEANGBZ","MEANGBH","MEANGBT"])].copy()
print("===== LATEST GRADIENT FEATURES, MONTHLY =====")
print(grad[["month","feature","transformed_median_training_iqr"]].to_string(index=False))


In [ ]:
# H1d-B: post-hoc Jan-Mar vs Apr-Aug 2026 internal boundary audit.
# This split is diagnostic because H1c showed the aggregate distance rise mainly from April onward.
period=np.full(len(ledger),"other",dtype=object)
period[(ledger["issue_dt"]>=pd.Timestamp("2026-01-01",tz="UTC")) & (ledger["issue_dt"]<pd.Timestamp("2026-04-01",tz="UTC"))]="2026_Jan_Mar"
period[(ledger["issue_dt"]>=pd.Timestamp("2026-04-01",tz="UTC")) & (ledger["issue_dt"]<pd.Timestamp("2026-09-01",tz="UTC"))]="2026_Apr_Aug"
ledger["h1d_period"]=period
full["h1d_period"]=period

boundary_rows=[]
for name,g in full[full["h1d_period"].isin(["2026_Jan_Mar","2026_Apr_Aug"])].groupby("h1d_period",sort=False):
    known=g[g["label_known"].astype(bool) & g["label"].isin([0,1])]
    row={
      "period":name,
      "cases":int(len(g)),
      "active_regions":int(g["region_component_id"].nunique()),
      "known_cases":int(len(known)),
      "positives":int(known["label"].sum()),
      "prevalence":float(known["label"].mean()) if len(known) else np.nan,
      "median_md2":float(g["physical_md2"].median()),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
    }
    for c in [x for x in full.columns if x.endswith("__latest")]:
        row[f"median::{c}"]=float(g[c].median())
    boundary_rows.append(row)
boundary=pd.DataFrame(boundary_rows)
boundary.to_csv(OUT/"boundary_2026_janmar_vs_apraug.csv",index=False)

# Raw latest feature medians for the same boundary.
bf=[]
for name in ["2026_Jan_Mar","2026_Apr_Aug"]:
    mask=ledger["h1d_period"].eq(name).to_numpy()
    for j,f in enumerate(features):
        raw=np.asarray(X[mask,2,j],dtype=float)
        bf.append({"period":name,"feature":f,"cases":int(mask.sum()),
                   "raw_median":float(np.median(raw)),
                   "raw_iqr":float(np.quantile(raw,.75)-np.quantile(raw,.25))})
pd.DataFrame(bf).to_csv(OUT/"boundary_2026_raw_feature_summary.csv",index=False)

print("===== 2026 INTERNAL BOUNDARY =====")
print(boundary.to_string(index=False))


In [ ]:
# H1d-C: compare late Cycle-24 quiet/transition years with 2026.
groups={
  "2018_2019": ledger["year"].isin([2018,2019]).to_numpy(),
  "2026": ledger["year"].eq(2026).to_numpy(),
}
quiet_rows=[]
for name,mask in groups.items():
    g=full.loc[mask]
    known=g[g["label_known"].astype(bool) & g["label"].isin([0,1])]
    row={
      "group":name,"cases":int(mask.sum()),"active_regions":int(g["region_component_id"].nunique()),
      "known_cases":int(len(known)),"positives":int(known["label"].sum()),
      "prevalence":float(known["label"].mean()) if len(known) else np.nan,
      "median_md2":float(g["physical_md2"].median()),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
    }
    for c in ["gradients__latest","inclination__latest","flux_pil__latest"]:
        row[f"median::{c}"]=float(g[c].median())
    quiet_rows.append(row)
quiet=pd.DataFrame(quiet_rows)
quiet.to_csv(OUT/"quiet_regime_comparison_2018_2019_vs_2026.csv",index=False)
print("===== 2018-2019 VS 2026 =====")
print(quiet.to_string(index=False))


In [ ]:
# H1d-D/E: leave-family/axis-out applicability sensitivity.
MPSV=[c for c in full.columns if "__" in c and not c.startswith("physical_")]
assert len(MPSV)==15
train_mask=full["role"].eq("train").to_numpy()

representations={
 "all_h1b":MPSV,
 "no_gradients_family":[c for c in MPSV if not c.startswith("gradients__")],
 "no_inclination_family":[c for c in MPSV if not c.startswith("inclination__")],
 "no_gradients_or_inclination":[c for c in MPSV if not (c.startswith("gradients__") or c.startswith("inclination__"))],
 "no_gradients_latest":[c for c in MPSV if c!="gradients__latest"],
}

def broad_regime(df):
    y=df["year"].to_numpy(int)
    out=np.full(len(df),"other",dtype=object)
    out[y<=2019]="cycle24_earlier"
    out[(y>=2021)&(y<=2025)]="cycle25_2021_2025"
    out[y==2026]="supplementary_2026"
    return out

reg=broad_regime(full)
leave_rows=[]
for rid,cols in representations.items():
    M=full[cols].to_numpy(float)
    lw=LedoitWolf().fit(M[train_mask])
    d=M-lw.location_
    md2=np.einsum("ij,jk,ik->i",d,lw.precision_,d)
    q99=float(np.quantile(md2[train_mask],.99))
    flag=md2>q99
    for grp in ["cycle24_earlier","cycle25_2021_2025","supplementary_2026"]:
        z=reg==grp
        leave_rows.append({
          "representation":rid,"axes":len(cols),"regime":grp,"cases":int(z.sum()),
          "training_q99":q99,
          "median_md2":float(np.median(md2[z])),
          "ood_rate_q99":float(flag[z].mean()),
        })
leave=pd.DataFrame(leave_rows)
leave.to_csv(OUT/"leave_family_out_applicability.csv",index=False)

print("===== LEAVE-FAMILY/AXIS-OUT APPLICABILITY =====")
print(leave.to_string(index=False))


In [ ]:
# H1d-F: source-lineage inventory.
cohorts=["cycle24","primary_cycle25","supplementary_2026"]
lineage={
 "build_receipt_sha256":sha256(SOURCE_ROOT/"build_receipt.json"),
 "independent_review_sha256":sha256(SOURCE_ROOT/"independent_review.json"),
 "build_receipt":json.loads((SOURCE_ROOT/"build_receipt.json").read_text()),
 "independent_review":json.loads((SOURCE_ROOT/"independent_review.json").read_text()),
 "cohorts":{},
}
low_card=[]

for cohort in cohorts:
    cp=SOURCE_ROOT/cohort
    idx_path=cp/"index.csv.gz"
    assert idx_path.is_file()
    idx=pd.read_csv(idx_path,low_memory=False)
    item={
      "index_sha256":sha256(idx_path),
      "rows":int(len(idx)),
      "columns":idx.columns.tolist(),
    }
    native=cp/"native_slot_ledger.csv.gz"
    if native.is_file():
        nd=pd.read_csv(native,low_memory=False)
        item["native_slot_ledger_sha256"]=sha256(native)
        item["native_slot_rows"]=int(len(nd))
        item["native_slot_columns"]=nd.columns.tolist()
        # Summarize likely lineage/version/source fields only when cardinality is manageable.
        for c in nd.columns:
            lc=c.lower()
            if any(k in lc for k in ["source","uri","path","series","product","version","segment","storage","provider"]):
                vals=nd[c].dropna().astype(str)
                nun=int(vals.nunique())
                if nun<=30:
                    for v,n in vals.value_counts().items():
                        low_card.append({"cohort":cohort,"table":"native_slot_ledger","column":c,"value":v,"count":int(n)})
                else:
                    low_card.append({"cohort":cohort,"table":"native_slot_ledger","column":c,"value":"<HIGH_CARDINALITY>","count":nun})
    # Index URI-prefix summary; informative for stored-history lineage only.
    for c in [x for x in idx.columns if "uri" in x.lower() or "path" in x.lower()]:
        vals=idx[c].dropna().astype(str)
        prefixes=vals.str.replace(r"/\d{4}/.*$","/<YEAR>/...",regex=True)
        for v,n in prefixes.value_counts().head(20).items():
            low_card.append({"cohort":cohort,"table":"index","column":c,"value":v,"count":int(n)})
    lineage["cohorts"][cohort]=item

(OUT/"source_lineage_inventory.json").write_text(json.dumps(lineage,indent=2,default=str)+"\n")
pd.DataFrame(low_card).to_csv(OUT/"source_lineage_low_cardinality.csv",index=False)

print("===== SOURCE LINEAGE FILES =====")
for cohort,item in lineage["cohorts"].items():
    print(cohort,item.get("rows"),item.get("native_slot_rows"),item.get("index_sha256"))


In [ ]:
receipt={
 "status":"PHASE_H1D_SOURCE_FEATURE_AUDIT_COMPLETE",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "later_regime_evidence":"posthoc_already_seen_in_A_to_G",
 "policy_action":"NONE",
 "diagnostics":[
   "raw_feature_monthly_2024_2026",
   "2026_Jan_Mar_vs_Apr_Aug",
   "2018_2019_vs_2026",
   "leave_family_out_applicability",
   "source_lineage_inventory",
 ],
 "source_sha256":{
   "sharp_tensor":sha256(SHARP),
   "sharp_ledger":sha256(LEDGER),
   "h1b_full":sha256(FULL),
   "h1b_transform":sha256(TRANSFORM),
 },
 "interpretation_status":"REQUIRES_REVIEW_BEFORE_H2",
}
(OUT/"h1d_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))
print("Outputs:",OUT)
